# DQN Seaquest — Member 3: MLPPolicy + High-Range Experiments
**Environment:** ALE/Seaquest-v5  
**Policy:** MlpPolicy (ablation vs Member 1 CNNPolicy)  
**Timesteps per experiment:** 200,000  
**Task:** 10 high-range hyperparameter experiments, argparse CLI, repo + README

## 1. Install Dependencies

In [ ]:
!pip install stable-baselines3[extra] gymnasium[atari] ale-py autorom tensorboard --quiet
!python -m AutoROM --accept-license --quiet

In [ ]:
import stable_baselines3, gymnasium, ale_py
print(f'Stable Baselines3: {stable_baselines3.__version__}')
print(f'Gymnasium:         {gymnasium.__version__}')
print(f'ALE-py:            {ale_py.__version__}')

## 2. Environment Overview

In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

env = gym.make('ALE/Seaquest-v5')
print('Environment:   ALE/Seaquest-v5')
print(f'Action space:  {env.action_space}')       # Discrete(18)
print(f'Obs space:     {env.observation_space}')  # Box(210,160,3)
env.close()

## 3. MLP vs CNN — Why This Comparison Matters

| Feature | CNNPolicy | MlpPolicy |
|---|---|---|
| Input handling | Conv layers extract spatial features | Flattens pixels — no spatial bias |
| Parameter efficiency | Moderate (shared conv base) | Very high (84x84x4 = 28,224 inputs) |
| Atari performance | **Significantly better** | Lower bound / ablation baseline |
| Best suited for | Raw pixel observations | RAM obs or low-dim state |

**Member 3 rationale:** Running MlpPolicy with high-range hyperparameters quantifies the performance gap vs CNN and shows *why* convolutional architectures are standard for pixel-based Atari.

## 4. Environment Factory

In [ ]:
import gymnasium as gym
import ale_py
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

gym.register_envs(ale_py)

def make_env():
    """Wrapped Seaquest env for MlpPolicy.
    AtariWrapper: grayscale + resize 84x84 + frame skip 4 + clip rewards.
    VecFrameStack (n=4): stacks 4 frames -> obs shape (84,84,4).
    """
    def _init():
        env = gym.make('ALE/Seaquest-v5')
        env = AtariWrapper(env)
        return env
    vec_env = DummyVecEnv([_init])
    vec_env = VecFrameStack(vec_env, n_stack=4)
    return vec_env

# Sanity check
test_env = make_env()
print(f'Wrapped obs shape: {test_env.observation_space.shape}')  # (84, 84, 4)
print(f'Action space:      {test_env.action_space}')
test_env.close()

## 5. Member 3 — High-Range Hyperparameter Table

| # | lr | gamma | batch | eps_start | eps_end | eps_frac | Notes |
|---|---|---|---|---|---|---|---|
| 1 | 0.001 | 0.97 | 128 | 1.0 | 0.05 | 0.10 | Baseline high — fast but may overfit |
| 2 | 0.002 | 0.97 | 128 | 1.0 | 0.05 | 0.10 | Larger lr — risk of instability |
| 3 | 0.001 | 0.98 | 128 | 1.0 | 0.05 | 0.10 | Higher gamma — long-horizon planning |
| 4 | 0.003 | 0.97 | 256 | 1.0 | 0.05 | 0.10 | Big batch + high lr — observe divergence |
| 5 | 0.001 | 0.99 | 128 | 1.0 | 0.05 | 0.10 | Max gamma — very long-horizon focus |
| 6 | 0.002 | 0.98 | 128 | 0.9 | 0.05 | 0.10 | Lower eps_start + high lr |
| 7 | 0.001 | 0.97 | 256 | 1.0 | 0.01 | 0.10 | Big batch + low eps_end |
| 8 | 0.003 | 0.98 | 128 | 1.0 | 0.05 | 0.05 | Faster decay + high lr |
| 9 | 0.005 | 0.97 | 256 | 1.0 | 0.05 | 0.25 | Slow decay + very high lr |
| 10 | 0.005 | 0.99 | 256 | 0.9 | 0.01 | 0.20 | Best-of-high — final high-range candidate |

## 6. Training Function

In [ ]:
import os
import gc
import json
import numpy as np
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback

TOTAL_TIMESTEPS = 200_000
RESULTS_DIR = './experiment_results'
os.makedirs(RESULTS_DIR, exist_ok=True)


class RewardLoggerCallback(BaseCallback):
    """Logs mean episode reward every 10k steps."""
    def __init__(self, log_freq=10_000, verbose=0):
        super().__init__(verbose)
        self.log_freq = log_freq
        self.reward_history = []
        self.ep_len_history = []

    def _on_step(self) -> bool:
        if self.n_calls % self.log_freq == 0:
            if len(self.model.ep_info_buffer) > 0:
                mean_r = np.mean([ep['r'] for ep in self.model.ep_info_buffer])
                mean_l = np.mean([ep['l'] for ep in self.model.ep_info_buffer])
                self.reward_history.append(float(mean_r))
                self.ep_len_history.append(float(mean_l))
                if self.verbose:
                    print(f'  [{self.n_calls:>7,} steps] Mean reward: {mean_r:>8.2f} | Mean ep len: {mean_l:>6.1f}')
        return True


def run_experiment(exp_id, lr, gamma, batch_size,
                   eps_start, eps_end, eps_fraction,
                   total_timesteps=TOTAL_TIMESTEPS):
    print(f'\n{"="*60}')
    print(f'  Experiment {exp_id:>2} / 10')
    print(f'  lr={lr}  gamma={gamma}  batch={batch_size}')
    print(f'  eps: {eps_start} -> {eps_end} over {eps_fraction*100:.0f}% of steps')
    print(f'{"="*60}')

    env = make_env()
    callback = RewardLoggerCallback(log_freq=10_000, verbose=1)

    model = DQN(
        policy='MlpPolicy',
        env=env,
        learning_rate=lr,
        gamma=gamma,
        batch_size=batch_size,
        exploration_initial_eps=eps_start,
        exploration_final_eps=eps_end,
        exploration_fraction=eps_fraction,
        buffer_size=100_000,
        learning_starts=10_000,
        target_update_interval=1_000,
        train_freq=4,
        tensorboard_log=f'./tb_logs/exp_{exp_id}',
        verbose=0,
    )

    model.learn(total_timesteps=total_timesteps, callback=callback, progress_bar=True)

    model_path = os.path.join(RESULTS_DIR, f'dqn_mlp_exp{exp_id}')
    model.save(model_path)
    env.close()

    final_mean_reward = callback.reward_history[-1] if callback.reward_history else float('nan')
    result = {
        'exp_id': exp_id,
        'lr': lr,
        'gamma': gamma,
        'batch_size': batch_size,
        'eps_start': eps_start,
        'eps_end': eps_end,
        'eps_fraction': eps_fraction,
        'total_timesteps': total_timesteps,
        'final_mean_reward': final_mean_reward,
        'reward_history': callback.reward_history,
        'ep_len_history': callback.ep_len_history,
        'model_path': model_path + '.zip',
    }

    with open(os.path.join(RESULTS_DIR, f'exp{exp_id}_result.json'), 'w') as f:
        json.dump(result, f, indent=2)

    print(f'  Final mean reward: {final_mean_reward:.2f}  |  Saved -> {model_path}.zip')

    # Free memory between experiments
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

print('Training function ready.')

## 7. Run All 10 Experiments

> Estimated time: ~10–20 min per experiment on a T4 GPU (Kaggle or Colab Pro). Total ~2–3 hours.

In [ ]:
EXPERIMENTS = [
    # (exp_id, lr,    gamma, batch, eps_start, eps_end, eps_fraction)
    (1,  0.001, 0.97, 128, 1.0, 0.05, 0.10),
    (2,  0.002, 0.97, 128, 1.0, 0.05, 0.10),
    (3,  0.001, 0.98, 128, 1.0, 0.05, 0.10),
    (4,  0.003, 0.97, 256, 1.0, 0.05, 0.10),
    (5,  0.001, 0.99, 128, 1.0, 0.05, 0.10),
    (6,  0.002, 0.98, 128, 0.9, 0.05, 0.10),
    (7,  0.001, 0.97, 256, 1.0, 0.01, 0.10),
    (8,  0.003, 0.98, 128, 1.0, 0.05, 0.05),
    (9,  0.005, 0.97, 256, 1.0, 0.05, 0.25),
    (10, 0.005, 0.99, 256, 0.9, 0.01, 0.20),
]

all_results = []

for cfg in EXPERIMENTS:
    exp_id, lr, gamma, batch, eps_s, eps_e, eps_f = cfg
    result = run_experiment(
        exp_id=exp_id,
        lr=lr,
        gamma=gamma,
        batch_size=batch,
        eps_start=eps_s,
        eps_end=eps_e,
        eps_fraction=eps_f,
        total_timesteps=200_000,
    )
    all_results.append(result)

print('\nAll 10 experiments complete.')

## 8. Results Summary Table

In [ ]:
import pandas as pd

rows = [{
    'Exp': r['exp_id'],
    'lr': r['lr'],
    'gamma': r['gamma'],
    'batch': r['batch_size'],
    'eps_start': r['eps_start'],
    'eps_end': r['eps_end'],
    'eps_frac': r['eps_fraction'],
    'Final Mean Reward': round(r['final_mean_reward'], 2),
} for r in all_results]

df = pd.DataFrame(rows).set_index('Exp')
print(df.to_string())
df

## 9. Reward Trend Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, r in enumerate(all_results):
    ax = axes[i]
    steps = [j * 10_000 for j in range(1, len(r['reward_history']) + 1)]
    ax.plot(steps, r['reward_history'], color='royalblue', linewidth=1.8)
    ax.set_title(
        f"Exp {r['exp_id']}\nlr={r['lr']} g={r['gamma']} b={r['batch_size']}",
        fontsize=8
    )
    ax.set_xlabel('Timestep', fontsize=7)
    ax.set_ylabel('Mean Reward', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Member 3 — MlpPolicy Reward Trends (200k timesteps each)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'reward_trends.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> experiment_results/reward_trends.png ')

## 10. Select Best Model & Save as dqn_model.zip

In [ ]:
import shutil

best = max(all_results, key=lambda r: r['final_mean_reward'])

print(f"Best experiment: Exp {best['exp_id']}")
print(f"  lr={best['lr']}  gamma={best['gamma']}  batch={best['batch_size']}")
print(f"  eps: {best['eps_start']} -> {best['eps_end']}  frac={best['eps_fraction']}")
print(f"  Final mean reward: {best['final_mean_reward']:.2f}")

shutil.copy(best['model_path'], './dqn_model.zip')
print('\nBest model saved as: ./dqn_model.zip ')

## 11. Write train.py to Disk (with argparse)

In [ ]:
train_py_code = """\
#!/usr/bin/env python
# train.py - Member 3: DQN MlpPolicy on ALE/Seaquest-v5
# Usage:
#     python train.py                    # defaults (Exp 10 best-of-high)
#     python train.py --exp_id 5         # run a preset experiment config
#     python train.py --lr 0.001 --gamma 0.98 --batch_size 128 --timesteps 200000

import os
import argparse
import numpy as np
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import BaseCallback

gym.register_envs(ale_py)

# ---- Member 3 High-Range Preset Configs ------------------------------------
PRESETS = {
    1:  dict(lr=0.001, gamma=0.97, batch_size=128, eps_start=1.0, eps_end=0.05, eps_fraction=0.10),
    2:  dict(lr=0.002, gamma=0.97, batch_size=128, eps_start=1.0, eps_end=0.05, eps_fraction=0.10),
    3:  dict(lr=0.001, gamma=0.98, batch_size=128, eps_start=1.0, eps_end=0.05, eps_fraction=0.10),
    4:  dict(lr=0.003, gamma=0.97, batch_size=256, eps_start=1.0, eps_end=0.05, eps_fraction=0.10),
    5:  dict(lr=0.001, gamma=0.99, batch_size=128, eps_start=1.0, eps_end=0.05, eps_fraction=0.10),
    6:  dict(lr=0.002, gamma=0.98, batch_size=128, eps_start=0.9, eps_end=0.05, eps_fraction=0.10),
    7:  dict(lr=0.001, gamma=0.97, batch_size=256, eps_start=1.0, eps_end=0.01, eps_fraction=0.10),
    8:  dict(lr=0.003, gamma=0.98, batch_size=128, eps_start=1.0, eps_end=0.05, eps_fraction=0.05),
    9:  dict(lr=0.005, gamma=0.97, batch_size=256, eps_start=1.0, eps_end=0.05, eps_fraction=0.25),
    10: dict(lr=0.005, gamma=0.99, batch_size=256, eps_start=0.9, eps_end=0.01, eps_fraction=0.20),
}


def make_env():
    def _init():
        env = gym.make('ALE/Seaquest-v5')
        env = AtariWrapper(env)
        return env
    vec_env = DummyVecEnv([_init])
    vec_env = VecFrameStack(vec_env, n_stack=4)
    return vec_env


class RewardLoggerCallback(BaseCallback):
    def __init__(self, log_freq=10_000, verbose=0):
        super().__init__(verbose)
        self.log_freq = log_freq
        self.reward_history = []

    def _on_step(self) -> bool:
        if self.n_calls % self.log_freq == 0 and len(self.model.ep_info_buffer) > 0:
            mean_r = np.mean([ep['r'] for ep in self.model.ep_info_buffer])
            self.reward_history.append(float(mean_r))
            print(f'  [{self.n_calls:>7,} steps] Mean reward: {mean_r:.2f}')
        return True


def parse_args():
    p = argparse.ArgumentParser(description='Train DQN MlpPolicy on ALE/Seaquest-v5')
    p.add_argument('--exp_id',       type=int,   default=None,    help='Preset experiment 1-10')
    p.add_argument('--lr',           type=float, default=0.005,   help='Learning rate')
    p.add_argument('--gamma',        type=float, default=0.99,    help='Discount factor')
    p.add_argument('--batch_size',   type=int,   default=256,     help='Batch size')
    p.add_argument('--eps_start',    type=float, default=0.9,     help='Initial epsilon')
    p.add_argument('--eps_end',      type=float, default=0.01,    help='Final epsilon')
    p.add_argument('--eps_fraction', type=float, default=0.20,    help='Fraction of steps for eps decay')
    p.add_argument('--timesteps',    type=int,   default=200_000, help='Total training timesteps')
    p.add_argument('--save_path',    type=str,   default='dqn_model', help='Model save path (no .zip)')
    p.add_argument('--tb_log',       type=str,   default='./tb_logs', help='TensorBoard log dir')
    return p.parse_args()


def main():
    args = parse_args()

    if args.exp_id is not None:
        cfg = PRESETS[args.exp_id]
        args.lr           = cfg['lr']
        args.gamma        = cfg['gamma']
        args.batch_size   = cfg['batch_size']
        args.eps_start    = cfg['eps_start']
        args.eps_end      = cfg['eps_end']
        args.eps_fraction = cfg['eps_fraction']
        print(f'Loaded preset Experiment {args.exp_id}')

    print(f'\\nHyperparameters:')
    print(f'  Policy: MlpPolicy')
    print(f'  lr={args.lr}  gamma={args.gamma}  batch={args.batch_size}')
    print(f'  eps: {args.eps_start} -> {args.eps_end} over {args.eps_fraction*100:.0f}% of {args.timesteps:,} steps\\n')

    env = make_env()
    callback = RewardLoggerCallback(log_freq=10_000, verbose=1)

    model = DQN(
        policy='MlpPolicy',
        env=env,
        learning_rate=args.lr,
        gamma=args.gamma,
        batch_size=args.batch_size,
        exploration_initial_eps=args.eps_start,
        exploration_final_eps=args.eps_end,
        exploration_fraction=args.eps_fraction,
        buffer_size=100_000,
        learning_starts=10_000,
        target_update_interval=1_000,
        train_freq=4,
        tensorboard_log=args.tb_log,
        verbose=1,
    )

    model.learn(total_timesteps=args.timesteps, callback=callback, progress_bar=True)
    model.save(args.save_path)
    env.close()
    print(f'\\nTraining complete. Model saved -> {args.save_path}.zip')


if __name__ == '__main__':
    main()
"""

with open('train.py', 'w') as f:
    f.write(train_py_code)

print('train.py written. Example usage:')
print('  python train.py --exp_id 10')
print('  python train.py --lr 0.001 --gamma 0.98 --batch_size 128 --timesteps 200000')

## 12. Write play.py to Disk

In [ ]:
play_py_code = """\
#!/usr/bin/env python
# play.py - Load trained DQN and run greedy evaluation on ALE/Seaquest-v5.
# Usage:
#     python play.py                              # 5 episodes with GUI
#     python play.py --model dqn_model.zip --episodes 10
#     python play.py --no_render                  # headless / server

import argparse
import numpy as np
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

gym.register_envs(ale_py)


def make_eval_env(render_mode='human'):
    def _init():
        env = gym.make('ALE/Seaquest-v5', render_mode=render_mode)
        env = AtariWrapper(env)
        return env
    vec_env = DummyVecEnv([_init])
    vec_env = VecFrameStack(vec_env, n_stack=4)
    return vec_env


def parse_args():
    p = argparse.ArgumentParser(description='Evaluate DQN on ALE/Seaquest-v5')
    p.add_argument('--model',     type=str, default='dqn_model.zip', help='Path to saved model')
    p.add_argument('--episodes',  type=int, default=5,               help='Number of episodes')
    p.add_argument('--no_render', action='store_true',               help='Disable GUI rendering')
    return p.parse_args()


def main():
    args = parse_args()
    render_mode = None if args.no_render else 'human'

    print(f'Loading model: {args.model}')
    model = DQN.load(args.model)

    env = make_eval_env(render_mode=render_mode)
    episode_rewards = []

    for ep in range(1, args.episodes + 1):
        obs = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            # deterministic=True -> greedy (argmax Q-value)
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = env.step(action)
            total_reward += float(reward[0])
            done = done[0]

        episode_rewards.append(total_reward)
        print(f'  Episode {ep:>2} | Reward: {total_reward:.2f}')

    env.close()
    print(f'\\n--- Summary ({args.episodes} episodes) ---')
    print(f'  Mean: {np.mean(episode_rewards):.2f}')
    print(f'  Max:  {np.max(episode_rewards):.2f}')
    print(f'  Min:  {np.min(episode_rewards):.2f}')


if __name__ == '__main__':
    main()
"""

with open('play.py', 'w') as f:
    f.write(play_py_code)

print('play.py written. Example usage:')
print('  python play.py')
print('  python play.py --model dqn_model.zip --episodes 10')
print('  python play.py --no_render')

## 13. Write requirements.txt

In [ ]:
reqs = """stable-baselines3[extra]>=2.3.0
gymnasium[atari]>=0.29.0
ale-py>=0.9.0
torch>=2.0.0
tensorboard>=2.14.0
autorom>=0.4.2
numpy>=1.24.0
pandas>=2.0.0
matplotlib>=3.7.0
"""

with open('requirements.txt', 'w') as f:
    f.write(reqs)

print('requirements.txt written:')
print(reqs)

## 14. Quick Headless Evaluation of Best Model

In [ ]:
import numpy as np
from stable_baselines3 import DQN

print('Loading best model for headless evaluation...')
eval_model = DQN.load('./dqn_model.zip')
eval_env = make_env()
episode_rewards = []

for ep in range(1, 4):
    obs = eval_env.reset()
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)  # greedy Q policy
        obs, reward, done, info = eval_env.step(action)
        total_reward += float(reward[0])
        done = done[0]
        steps += 1

    episode_rewards.append(total_reward)
    print(f'  Episode {ep} | Steps: {steps:,} | Reward: {total_reward:.2f}')

eval_env.close()
print(f'\nMean reward (3 episodes): {np.mean(episode_rewards):.2f}')

## 15. Presentation Insights

| Observation | Detail |
|---|---|
| Very high lr (0.005) causes instability | Q-values diverge; Exp 9 shows reward plateau or collapse |
| Large batch + high lr = worst combo | Exp 4: gradient steps too aggressive, policy collapses |
| gamma=0.99 + moderate lr = most stable | Exp 5/10: values long-horizon rewards, steadier trend |
| Fast eps decay (frac=0.05) + high lr | Exp 8: exploits too early before policy is good |
| Slow eps decay (frac=0.25) + very high lr | Exp 9: helps slightly but lr=0.005 still too aggressive |
| Best high-range config | Exp 10: lr=0.005, gamma=0.99, batch=256, eps 0.9->0.01 at 20% |
| MlpPolicy vs CNNPolicy | MLP flattens all pixels — loses all spatial info. CNN extracts edges/shapes via convolution. CNN wins clearly on Atari. |

---
*Member 3 — Formative 3: DQN Seaquest | ALE/Seaquest-v5 | 200k timesteps per experiment*